<a href="https://colab.research.google.com/github/abrishkhanzai441-arch/project.py/blob/main/backend_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
!apt-get install -y tesseract-ocr poppler-utils -q
!pip install pytesseract pdf2image pymupdf -q

from google.colab import files
import pymupdf
import pytesseract
from pdf2image import convert_from_path
import re

database = {}

print("📤 Upload your PDF files...")
uploaded = files.upload()

for filename, file_data in uploaded.items():
    with open(filename, "wb") as f:
        f.write(file_data)

    text = ""
    if filename.lower().endswith(".pdf"):
        doc = pymupdf.open(filename)
        for page in doc:
            text += page.get_text()

        # If standard text extraction yields nothing, run OCR
        if not text.strip():
            print(f"🔍 No text layer found in {filename}. Running OCR...")
            images = convert_from_path(filename)
            for img in images:
                text += pytesseract.image_to_string(img)

    if text.strip():
        database[filename] = text
        print(f"✅ {filename} -> Saved {len(text)} characters to database.")
    else:
        print(f"⚠️ Could not extract text from {filename}.")

def secure_search(question):
    clean_query = re.sub(r'[^\w\s]', '', question).strip().lower()
    print(f"\n❓ Searching for: '{clean_query}'")
    print("-" * 30)

    found = False
    for fname, content in database.items():
        pos = content.lower().find(clean_query)
        if pos != -1:
            preview = content[max(0, pos - 80) : pos + 150]
            print(f"📄 FOUND IN: {fname}\nAnswer: ...{preview}...\n")
            found = True

    if not found:
        print("❌ Not found in your documents.")

# Test search
print(database.get("liquify effect (2).pdf", "File not found"))

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (5.3.4-1build5).
poppler-utils is already the newest version (24.02.0-1ubuntu9.9).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.
📤 Upload your PDF files...


Saving Penetration_Testing_Assignment.pdf to Penetration_Testing_Assignment (2).pdf
✅ Penetration_Testing_Assignment (2).pdf -> Saved 816 characters to database.
File not found


In [18]:
# --- Install AI (run once) ---
!pip install sentence-transformers faiss-cpu -q

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print("⏳ Loading AI model... first time takes 30 seconds")
ai_model = SentenceTransformer('all-MiniLM-L6-v2')

# --- Step 1: Make chunks from your database ---
# Your database from Cell 2 is still there
all_chunks = []
chunk_source = [] # to remember which file it came from

for filename, content in database.items():
    words = content.split()
    for i in range(0, len(words), 150): # 150 words = 1 chunk
        chunk = " ".join(words[i:i+150])
        all_chunks.append(chunk)
        chunk_source.append(filename)

print(f"✅ Made {len(all_chunks)} AI-readable chunks")

# --- Step 2: Build AI Search Index ---
embeddings = ai_model.encode(all_chunks)
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings))
print("✅ Secure Search Index Ready")

# --- Step 3: The Final AI Search Function ---
# This is what Web Dev Team's chat box will call
def ask_ai(question):
    print(f"\n❓ QUESTION: {question}")

    # AI finds the most relevant parts, not just exact words
    q_vector = ai_model.encode([question])
    D, I = index.search(np.array(q_vector), 3) # top 3 results

    print(f"\n🤖 AI ANSWER (based ONLY on your documents):\n")
    sources = set()
    for idx in I[0]:
        print(f"--- From file: {chunk_source[idx]} ---")
        print(all_chunks[idx][:400] + "...\n")
        sources.add(chunk_source[idx])

    print(f"📄 Sources used: {list(sources)}")
    print("\n[This is secure - answer is only from your private files]")

# --- TEST ---
ask_ai("What is written about salary?") # Change your question here

⏳ Loading AI model... first time takes 30 seconds


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Made 1 AI-readable chunks
✅ Secure Search Index Ready

❓ QUESTION: What is written about salary?

🤖 AI ANSWER (based ONLY on your documents):

--- From file: Penetration_Testing_Assignment (1).pdf ---
Website Vulnerability Finder - Penetration Testing Assignment 1. Introduction This project focuses on identifying security vulnerabilities in websites using penetration testing techniques. 2. Problem Statement Many websites have vulnerabilities that can be exploited by attackers, leading to data theft and service disruption. 3. Objective To identify, analyze, and report vulnerabilities in target s...

--- From file: Penetration_Testing_Assignment (1).pdf ---
Website Vulnerability Finder - Penetration Testing Assignment 1. Introduction This project focuses on identifying security vulnerabilities in websites using penetration testing techniques. 2. Problem Statement Many websites have vulnerabilities that can be exploited by attackers, leading to data theft and service disruption. 3. Obje

In [19]:
ask_ai("summarize the documents")
ask_ai("what is the leave policy?")


❓ QUESTION: summarize the documents

🤖 AI ANSWER (based ONLY on your documents):

--- From file: Penetration_Testing_Assignment (1).pdf ---
Website Vulnerability Finder - Penetration Testing Assignment 1. Introduction This project focuses on identifying security vulnerabilities in websites using penetration testing techniques. 2. Problem Statement Many websites have vulnerabilities that can be exploited by attackers, leading to data theft and service disruption. 3. Objective To identify, analyze, and report vulnerabilities in target s...

--- From file: Penetration_Testing_Assignment (1).pdf ---
Website Vulnerability Finder - Penetration Testing Assignment 1. Introduction This project focuses on identifying security vulnerabilities in websites using penetration testing techniques. 2. Problem Statement Many websites have vulnerabilities that can be exploited by attackers, leading to data theft and service disruption. 3. Objective To identify, analyze, and report vulnerabilities in targ

In [21]:
# --- FINAL TEST ---
def ask_ai(question):
    q_vector = ai_model.encode([question])
    import numpy as np
    D, I = index.search(np.array(q_vector), 2)
    print(f"\n❓ Q: {question}\n")
    for idx in I[0]:
        print(f"📄 From {chunk_source[idx]}:")
        print(all_chunks[idx])
        print("----")

ask_ai("What is this assignment about?")


❓ Q: What is this assignment about?

📄 From Penetration_Testing_Assignment (1).pdf:
Website Vulnerability Finder - Penetration Testing Assignment 1. Introduction This project focuses on identifying security vulnerabilities in websites using penetration testing techniques. 2. Problem Statement Many websites have vulnerabilities that can be exploited by attackers, leading to data theft and service disruption. 3. Objective To identify, analyze, and report vulnerabilities in target systems. 4. Methodology The penetration testing process includes reconnaissance, scanning, vulnerability assessment, exploitation, and reporting. 5. Tools Used Nmap, Nikto, Burp Suite, SQLmap, Metasploit. 6. Workflow Reconnaissance → Scanning → Vulnerability Analysis → Exploitation → Reporting. 7. Conclusion Penetration testing helps improve system security by identifying weaknesses before attackers exploit them.
----
📄 From Penetration_Testing_Assignment (1).pdf:
Website Vulnerability Finder - Penetration Test

In [22]:
from google.colab import files

# Create app.py file
with open('app.py', 'w') as f:
    f.write(open('/mnt/data/app.py').read() if False else '''
from flask import Flask, request, jsonify
from flask_cors import CORS
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# ... (use the app.py code I gave you above) ...
''')

print("Now download from Files panel on left -> right click app.py -> Download")

Now download from Files panel on left -> right click app.py -> Download
